In [ ]:
import Implementation_OFLP as IO
import run_OFLP as rO
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from tqdm import tqdm
import numpy as np
a = rO.read_instance('ofl_9_100i.gms')


def get_feas_points(circles_used : list, x_lb, x_ub, y_lb, y_ub):
    
    # We now look at all of the intersections between the circles and the square
    n = len(circles_used)
    m = len(circles_used) + 4
    points = []
    starting_p = [[x_lb, y_lb], [x_lb, y_ub], [x_ub, y_lb], [x_ub, y_ub]]
    for p in starting_p:
        p_inter = False
        for c in circles_used:
            if IO.interior_circle(p,c):
                p_inter = True
        if not p_inter:
            points.append(p)
    
    points_belong_circle = [[]]*m
    for i in range(len(circles_used)):
        point, points_belong_circle = IO.box_circles(x_lb, x_ub, y_lb, y_ub, circles_used, i, points_belong_circle)
        #print(f"We are looking at the circle: {circle} and it produces points :{point}")
        for p in point:
            #print(f"Point {p} is not in the list of points: {points}? {p not in points}")
            if type(p) != list:
                n_point = [p]
                p = n_point
            interior = False
            for c in circles_used:
                if IO.interior_circle(p,c):
                    interior = True
            if not interior:
                if p not in points_belong_circle[i+4]:
                    points_belong_circle[i+4].append(p)
                if p not in points:
                    points.append(p)


    # We now look at all of the intersections between the circles
    for i in range(n):
        for j in range(i + 1,n):
            c1, c2 = circles_used[i], circles_used[j]
            if c1 == c2:
                continue
            elif ((c1[0] - c2[0]) ** 2) + ((c1[1] - c2[1]) ** 2) >= c1[2] ** 2 + c2[2] ** 2:
                continue
            else:
                intersections = IO.circles_circles(circles_used,i,j)
            for point in intersections:
                if type(point) != list:
                    n_point = [point]
                    point = n_point
                if point not in points:
                    interior = False
                    if (x_lb <= point[0] <= x_ub) and (y_lb <= point[1] <= y_ub):
                        for c in circles_used:
                            if IO.interior_circle(point, c) == True:
                                interior = True
                        if (not interior):
                            points_belong_circle[i+4].append(point)
                            points_belong_circle[j+4].append(point)
                            points.append(point)
    return points

def is_feas_solution(points, fac_fac_dist, facilities) -> float:
    G = nx.Graph()
    n = len(points)
    for i in range(n):
        for j in range(i+1, n):
            if rO.distance(points[i], points[j]) >= fac_fac_dist:
                G.add_edge(i, j)
    return nx.max_weight_clique(G, weight=None)


def create_init_sol(facilities, communities, x_lb, x_ub, y_lb, y_ub, fac_fac_dist, r_guess : float=0) -> float:
    # The idea of the algorithm is to decrease the radius until we find a feasible radius, this way we create a good intial solution.
    feas_sol_found = False
    max_iter = 1e3 * len(communities)
    t = 0
    if r_guess == 0:
        r = 25 / len(communities)
    else:
        r = r_guess
    with tqdm(total=max_iter, desc="Shrinking r") as pbar:
        while (not feas_sol_found) and (t < max_iter):
            print(t)
            circles = []
            for i in communities:
                circles.append([i[0], i[1], r])
            points = get_feas_points2(circles, x_lb, x_ub, y_lb, y_ub)
            # We now try to fit the facilities in the points while maintaining the fac_fac_distance
            feas = is_feas_solution(points, fac_fac_dist, facilities)
            if feas[1] < facilities:
                r *= 0.95
                t += 1
            else:
                feas_sol_found = True
                fig, ax = plt.subplots()
                box = patches.Rectangle((x_lb, y_lb), x_ub-x_lb, y_ub - y_lb, linewidth=3, edgecolor='black', facecolor='none')
                ax.add_patch(box)
                for c in circles:
                    circle = patches.Circle((c[0], c[1]), c[2], linewidth=1, edgecolor='black', fill = True)
                    ax.add_patch(circle)
                for p in feas[0]:
                    ax.scatter(points[p][0], points[p][1], s=20, c='red', marker='o')
                # Set limits and equal aspect ratio
                ax.set_xlim(x_lb - abs(x_lb-x_ub)/2, x_ub + abs(x_lb-x_ub)/2)
                ax.set_ylim(y_lb - abs(y_lb-y_ub)/2, y_ub + abs(y_lb-y_ub)/2)
                ax.set_aspect('equal')
            
                # Show the plot
                plt.grid(True)
                plt.show()
                return r, feas
    return r, feas


create_init_sol(a['facilities'], a['communities'], a['x_bounds'][0], a['x_bounds'][1], a['y_bounds'][0], a['y_bounds'][1], a['ff_sq_bounds'][0])

Shrinking r:   0%|          | 0/100000.0 [00:00<?, ?it/s]

In [35]:
def get_feas_points2(circles, x_lb, x_ub, y_lb, y_ub) -> float:
    # We create 1000 possible facility locations, we then check for each which are feasible to get points to try to fit the number of facilities.
    poss_fac = []
    x_coords = np.linspace(x_lb,x_ub,1000)
    y_coords = np.linspace(y_lb,y_ub,1000)
    for x in x_coords:
        for y in y_coords:
            poss_fac.append([x,y])
    points = []
    for p in poss_fac:
        feas = True
        for c in circles:
            if IO.interior_circle(p,c):
                feas = False
                break
        if not feas:
            points.append(p)
    return points